# Präambel

In [1]:
from sympy import *
from IPython.display import display, Math, Latex, Markdown
#init_session()
init_printing(use_unicode=True)

In [2]:
from dataclasses import dataclass
from decimal import Decimal
from functools import partial

@dataclass
class Einwirkungen:
    N: float = None
    V: float = None
    My: float = None
    Mz: float = None


@dataclass
class Querschnitt:
    querschnitt: str
    A: float
    Avy: float
    Avz: float
    b: float
    h: float
    Iy: float
    Iz: float
    iy: float
    iz: float
    KLy: str
    KLz: str

    def um_achse(self, attr, achse):
        return self.__getattribute__(f'{attr}{achse}')

    I = lambda self, achse: self.um_achse('I', achse)
    i = lambda self, achse: self.um_achse('i', achse)
    KL = lambda self, achse: self.um_achse('KL', achse)
    Av = lambda self, achse: self.um_achse('Av', achse)

    
    
@dataclass
class Stahlguete:
    guete: str
    E: int
    fy: float
    fu: float
    
@dataclass
class Stab(Querschnitt, Stahlguete):
    l: float
    Lcry: float = None
    Lcrz: float = None

    def __init__(self, guete, QS, **kwargs):
        for k,v in kwargs.items():
            self.__setattr__(k, v)
            
        Querschnitt.__init__(self, **QS.__dict__)
        Stahlguete.__init__(self, **guete.__dict__)
        
    def um_achse(self, attr, achse):
        return self.__getattribute__(f'{attr}{achse}')
        
    Lcr = lambda self, achse: self.um_achse('Lcr', achse)

S235 = Stahlguete(guete='S235', E=21000, fy=23.5, fu=36)

HEB160 = Querschnitt(
    querschnitt='HEB 160',
    A   = 54.25,
    Avy = 41.6,
    Avz = 12.65,
    b   = 16,
    h   = 16,
    Iy  = 2492,
    Iz  = 889.2,
    iy  = 6.78,
    iz  = 4.05,
    KLy = 'b',
    KLz = 'c',
)

HEB240 = Querschnitt(
    querschnitt='HEB 240',
    A   = 106,
    Avy = 81.6,
    Avz = 33.23,
    b   = 24,
    h   = 24,
    Iy  = 11260,
    Iz  = 3923,
    iy  = 10.31,
    iz  = 6.08,
    KLy = 'b',
    KLz = 'c',
)

IPE200 = Querschnitt(
    querschnitt='IPE 200',
    A   = 28.50,
    Avy = 17,
    Avz = 14,
    b   = 10,
    h   = 20,
    Iy  = 1940,
    Iz  = 142,
    iy  = 8.26,
    iz  = 2.24,
    KLy = 'a',
    KLz = 'b',
)
HEB650 = Querschnitt(
    querschnitt='HEB 650',
    A   = 286,
    Avy = 186,
    Avz = 122.04,
    b   = 30,
    h   = 65,
    Iy  = 210600,
    Iz  = 13980,
    iy  = 27.1,
    iz  = 6.99,
    KLy = 'a',
    KLz = 'b',
)

pfette  = Stab(S235, HEB160, l = 2)
pfette

Stab(guete='S235', E=21000, fy=23.5, fu=36, querschnitt='HEB 160', A=54.25, Avy=41.6, Avz=12.65, b=16, h=16, Iy=2492, Iz=889.2, iy=6.78, iz=4.05, KLy='b', KLz='c', l=2, Lcry=None, Lcrz=None)

In [3]:
def md(s):
    display(Markdown(s))
    
def ausnutzung(p):
    color = '#000'
    emoji = ''
    if p > 1:
        color = '#f00'
        emoji = '🔴'
    elif p > 0.7:
        color = '#0f0'
        emoji = '🟢'
    elif p < 0.7:
        color = '#00f'
        emoji = '🟡'
    #md(f'#### <span style="color:{color};">{int(p*100)}%</span>')
    md(f'#### {emoji} {int(N(p, 2)*100)}%')
    return p


def note(s):
    md(f'<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">{s}</span></div>')
ausnutzung(13.37)
ausnutzung(0.238)
N(0.238, 2) * 100

#### 🔴 1338%

#### 🟡 23%

# Nachweis-Funktionen

## Querschnitte

### Zug

In [4]:
def zug(stab, N_Ed, A_net=None):
    γ_M0 = 1
    γ_M2 = 1.25
    N_plRd = (stab.A * stab.fy)/γ_M0
    N_uRd = (0.9 * A_net * stab.fu)/γ_M2 if A_net else oo

    return ausnutzung(N(N_Ed/min(N_plRd, N_uRd), 3))

### Druck

In [5]:
def druck(stab, N_Ed):
    γ_M0 = 1
    N_cRd = (stab.A * stab.fy)/γ_M0
    
    return ausnutzung(N(N_Ed/N_cRd, 3))

### Biegung

In [6]:
def biegung(um_achse, stab, M_Ed):
    γ_M0 = 1
    h = stab.h if um_achse == 'y' else stab.b
    W_elmin = stab.I(um_achse)/(h/2)
    M_elRd = (W_elmin * stab.fy)/γ_M0
    
    return ausnutzung(N(M_Ed/M_elRd, 3))

biegung_y = partial(biegung, 'y')
biegung_z = partial(biegung, 'z')

### Querkraft

In [7]:
def querkraft(um_achse, stab, V_Ed):
    γ_M0 = 1
    V_plRd = stab.Av(um_achse) * (f_y/(sqrt(3) * γ_M0))
    
    return ausnutzung(N(V_Ed/V_plRd, 3))

querkraft_y = partial(querkraft, 'y')
querkraft_z = partial(querkraft, 'z')

### Kombi: M + N

In [8]:
def kombi_M_N(um_achse, stab, N_Ed, V_Ed, M_Ed):
    γ_M0 = 1
    V_plRd = stab.Av(um_achse) * (f_y/(sqrt(3) * γ_M0))
    if (V_Ed / V_plRd) <= 0.5:
        md('Nachweis nicht Notwendig')
        return
    
    return ausnutzung(druck(stab, N_Ed) + biegung(um_achse, stab, M_Ed))

## Bauteil

In [9]:
γ_M1 = 1 #TODO tripple-check

def λ1(stab):
    return pi * sqrt(stab.E/stab.fy)

def λxz(um_achse, stab):
    return stab.Lcr(um_achse)/(stab.i(um_achse) * λ1(stab))

def χxz(um_achse, stab):
    λ = λxz(um_achse, stab)
    α = {'a0': 0.13, 'a': 0.21, 'b': 0.34, 'c': 0.49, 'd': 0.76}.get(stab.KL(um_achse))
    Φ = 0.5* (1 + α * (λ-0.2) + λ**2)
    return min(1, 1/(Φ + sqrt(Φ**2 - λ**2)))

### Biegeknicken

In [34]:
def biege_knicken(um_achse, stab, last: Einwirkungen):
    assert(stab.Lcr(um_achse))
    χ = χxz(um_achse, stab)
    N_Rd = stab.A * (stab.fy/γ_M1)
    W_y = stab.Iy/(stab.h * 0.5) ## CHECK past?
    My_Rd =  W_y * (stab.fy / γ_M1) # W_y passt?

    n_y = last.N / (χ * N_Rd)
    k_y = min((1 + (λxz(um_achse, stab) - 0.2) * n_y),
              (1 + 0.8 * n_y),
             )
    α = 0.6 if um_achse == 'z' else 1

    #C_my = max(0.6 + 0.4 * ψ,
    #           0,4)
    C_my = 0.9 # XXX auf sicheren seite(?)
    res = (last.N/(χ * N_Rd)) + (α * k_y * (C_my * last.My)/My_Rd)
    return ausnutzung(N(res, 5))

biege_knicken_y = partial(biege_knicken, 'y')
biege_knicken_z = partial(biege_knicken, 'z')

# Stäbe

## 1 Pfette

### Biegedrillknicken (um y)

In [11]:
# TODO

### Querschnitt (Druck)

In [12]:
# TODO

## 2 Binder Obergurt

In [13]:
obergurt = Stab(S235, HEB240, l=250.3, Lcry=250.3, Lcrz=250.3)

OG_N_Edmax = 1292 # Schnee leitend + Wind(0)
OG_N_Edmin = 176  # Wind (180)
OG_My_max = 6.87  # Wind (180)
OG_My_min = 51.56 # Schnee leitend + wind(0)

### Querschnitt (Zug)

In [14]:
zug(obergurt, OG_N_Edmax)

#### 🟡 51%

### Querschnitt (Druck)

In [15]:
druck(obergurt, OG_N_Edmin)

#### 🟡 7%

### Knicken in Ebene (y)

In [16]:
biege_knicken_y(obergurt, Einwirkungen(N=OG_N_Edmin, My=OG_My_min))

#### 🟡 7%

### Knicken aus der Ebene (z)

In [17]:
biege_knicken_z(obergurt, Einwirkungen(N=OG_N_Edmin, My=OG_My_min))

#### 🟡 8%

### Biegung

In [18]:
biegung_y(obergurt, M_Ed=OG_My_min)

#### 🟡 0%

### Querkraft

## 3 Binder Untergurt

In [33]:
## Knickhalterung durch Fuellstaebe in Ebene
#NB.: Laengsten stab mit hoechster N(x) kombiniert (sichere seite)
untergurt_krag_y = Stab(S235, HEB240, l=256)

## Aus Ebene gibts keine Knicksicherung, gestamter stab wirkt
untergurt_krag_z = Stab(S235, HEB240, l=1134.5, Lcrz=2*1134.5)

### Querschnitt (Zug)

In [20]:
# TODO

### Querschnitt (Druck)

In [21]:
# TODO

### Knicken in Ebene

In [22]:
# TODO

### Knicken aus der Ebene

In [32]:
l_cr = untergurt_krag_z.l * 2
#l_cr = 660
biege_knicken_z(untergurt_krag_z, Einwirkungen(N=1377, My=0))
note('Veraenderliche N(x) -- maximaler wert notwendig? ._.')
note('TODO: ggf knick-halterung bei ~6.5m von Stuetze B anbringen')

#### 🔴 101%

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">Veraenderliche N(x) -- maximaler wert notwendig? ._.</span></div>

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">TODO: ggf knick-halterung bei ~6.5m von Stuetze B anbringen</span></div>

### ⁉️ BDK?

### Biegung

In [ ]:
# TODO

## 4 Binder Füllstab

In [104]:
fuellstab_druck = Stab(S235, IPE200, l=185)
fuellstab_zug = Stab(S235, IPE200, l=192)

### Querschnitt (Zug)

In [ ]:
# TODO

### Knicken in Ebene

In [135]:
## Druck: Knicken aus Ebene
# Maßgeblicher Stab:
# LF: Schnee leitend + Wind (0)
biege_knicken_y(fuellstab_druck, l_cr=fuellstab_druck.l, N_Ed=290)

#### 🟡 43%

### Knicken aus der Ebene

In [136]:
## Druck: Knicken aus Ebene
# Maßgeblicher Stab:
# LF: Schnee leitend + Wind (0)
biege_knicken_z(fuellstab_druck, l_cr=fuellstab_druck.l, N_Ed=290)

#### 🟡 64%

## 5 Dachverband

### Querschnitt (Zug)

In [ ]:
# TODO

## 6 Wandverband

### Querschnitt (Zug)

In [ ]:
# TODO

## 7 Stütze A

In [167]:
stuetzeA = Stab(S235, HEB160, l=650)

CA_N_Edmax = 356 # Schnee leitend + Wind(0)
CA_N_Edmin = 55  # Wind (180)


### Querschnitt (Zug)

In [ ]:
# TODO

### Querschnitt (Druck)

In [ ]:
# TODO

### Knicken in Ebene

In [171]:
biege_knicken_y(stuetzeA, l_cr=stuetzeA.l, N_Ed=CA_N_Edmin)

#### 🟡 7%

### Knicken aus der Ebene

In [169]:
biege_knicken_z(stuetzeA, l_cr=stuetzeA.l, N_Ed=CA_N_Edmin)

#### 🟡 16%

## 8 Stütze B

### ⁉️ BDK?

In [167]:
stuetzeB = Stab(S235, HEB650, l=625)

CB_N_Edmax = 95   # Wind (180)
CB_N_Edmin = 847  # Schnee leitend + Wind(0)


### Querschnitt (Zug)

In [ ]:
# TODO

### Querschnitt (Druck)

In [ ]:
# TODO

### Knicken in Ebene

In [ ]:
# TODO

### Knicken aus der Ebene

In [ ]:
# TODO

### Biegung

In [ ]:
# TODO